# Notebook 12: the organisers' fifteen pretrained encoders

`openadmet/chemprop-foundation-pretraining-weights` holds fifteen ChemProp checkpoints,
Apache 2.0, published on the day the challenge opened. Each was pretrained against a
**different molecular representation**: ECFP, atom-pair, ERG, Jazzy, Minimol, Osmordred
and three PCA variants of it, PubChem, RDKit2D, a surrogate ADME model, and four
three-dimensional ones, `e3fp`, `usrcat` and `whim`.

This matters because of the one thing this project has established most clearly.
Four networks trained from the same checkpoint, varying seeds, heads and auxiliary data,
all correlate above 0.95 and none adds anything to the blend. One network from a
*different* checkpoint correlates 0.70 to 0.90 and earns weight 0.25 while being 0.046
worse on its own. Every encoder we have comes from one repository with two files in it.
Here are fifteen more, each deliberately built to see molecules differently.

The plan is deliberately cheap before it is expensive:

1. **Probe.** Try to load all fifteen and report which work. Two minutes, no training.
2. **Embed.** Extract frozen embeddings from the ones that load, about 2.5 minutes each.
3. **Select.** Measure how much each disagrees with `chemprop_medium` and with the others,
   and carry forward only the genuinely different ones.
4. **Fit.** TabPFN on the selected embeddings, which is our strongest arm type.

Step 3 is the point. Fifteen more near-duplicates would be worth nothing, and we can find
that out from the embeddings alone before spending an hour fitting them.

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout or 'NO GPU')

In [ ]:
PAYLOAD_FILE = '/content/drive/MyDrive/cyp_payloads/tiny_6387286b00.tgz'

import os, tarfile
os.makedirs('/content/cache', exist_ok=True)
from google.colab import drive
drive.mount('/content/drive')
with tarfile.open(PAYLOAD_FILE) as t:
    t.extractall('/content/cache')
OUTDIR='/content/results'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTDIR='/content/drive/MyDrive/cyp_results'
except Exception as e:
    print('Drive unavailable:', type(e).__name__)
os.makedirs(OUTDIR, exist_ok=True)
print('writing to', OUTDIR)

In [ ]:
!pip -q install chemprop==2.3.1

In [ ]:
import json, time, numpy as np, pandas as pd, torch
from pathlib import Path
from scipy.stats import spearmanr
from huggingface_hub import hf_hub_download
from chemprop import data, featurizers
from chemprop.models.model import MPNN

CACHE=Path('/content/cache'); OUT=Path(OUTDIR)
ISO=['CYP1A2','CYP2C9','CYP2D6','CYP3A4']
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
meta=pd.read_csv(CACHE/'meta_train.csv.gz'); mete=pd.read_csv(CACHE/'meta_test.csv.gz')
hold=np.load(CACHE/'holdout_mask.npy')
G=np.load(CACHE/'scaffold_groups.npy', allow_pickle=True)
QH=np.load(CACHE/'qhts_tr.npy'); QH_TE=np.load(CACHE/'qhts_te.npy')
AUX=np.load(CACHE/'aux_train.npy'); AUX_TE=np.load(CACHE/'aux_test.npy')
SM_TR=meta['SMILES'].tolist(); SM_TE=mete['SMILES'].tolist()
REPO='openadmet/chemprop-foundation-pretraining-weights'
NAMES=['e3fp','usrcat','whim','rdkit2d','atompair','minimol','surrogate_adme',
       'ecfp','erg','jazzy','pubchem','osmordred','osmordred_pca80',
       'osmordred_pca90','osmordred_pca95']
print('device', DEVICE, '| candidates', len(NAMES))

## Step 1, probe

Nothing is trained here. Each checkpoint is downloaded and handed to chemprop's loader,
and anything that refuses is reported and dropped.

In [ ]:
OK={}
for n in NAMES:
    try:
        p=hf_hub_download(REPO, f'{n}__s42.pt')
        m=MPNN.load_from_file(p)
        OK[n]=p
        print(f'  {n:18s} loaded')
    except Exception as e:
        print(f'  {n:18s} FAILED  {type(e).__name__}: {str(e)[:70]}')
print(f'\n{len(OK)} of {len(NAMES)} usable')
assert OK, 'none of the checkpoints load; stop here'

## Step 2, frozen embeddings

In [ ]:
FZ=featurizers.SimpleMoleculeMolGraphFeaturizer()
def embed(ckpt, smiles, bs=256):
    m=MPNN.load_from_file(ckpt); m.eval()
    pts=[data.MoleculeDatapoint.from_smi(s,[0.0]) for s in smiles]
    ld=data.build_dataloader(data.MoleculeDataset(pts,FZ), num_workers=0,
                             batch_size=bs, shuffle=False, drop_last=False)
    out=[]
    with torch.inference_mode():
        for b in ld:
            bmg,V_d,X_d,*_=b
            out.append(m.fingerprint(bmg,V_d,X_d).numpy())
    r=np.concatenate(out,0)
    assert len(r)==len(smiles), f'{len(r)} embedded, {len(smiles)} expected'
    return r.astype(np.float32)

EMB={}
t0=time.time()
for n,p in OK.items():
    f=OUT/f'fnd_{n}_train.npy'
    if f.exists():
        EMB[n]=(np.load(f), np.load(OUT/f'fnd_{n}_test.npy'))
        print(f'  {n:18s} cached {EMB[n][0].shape}'); continue
    try:
        a=embed(p, SM_TR); b=embed(p, SM_TE)
        np.save(f,a); np.save(OUT/f'fnd_{n}_test.npy', b)
        EMB[n]=(a,b)
        print(f'  {n:18s} {a.shape}  ({time.time()-t0:.0f}s)', flush=True)
    except Exception as e:
        print(f'  {n:18s} embedding failed: {type(e).__name__}: {str(e)[:60]}', flush=True)
print(f'\n{len(EMB)} embeddings ready')

## Step 3, which of them actually disagree

A cheap proxy before any fitting: fit a ridge from each embedding onto CYP3A4 on
development rows, and correlate the resulting predictions. Encoders whose predictions
correlate above about 0.95 with one another are near-duplicates and only one is worth
carrying.

In [ ]:
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import cross_val_predict
y=meta['CYP3A4_pIC50_direct_inhibition'].to_numpy(float)
dev=np.flatnonzero(np.isfinite(y) & ~hold)
proxy={}
for n,(a,_) in EMB.items():
    X=np.nan_to_num(a[dev])
    p=cross_val_predict(RidgeCV(alphas=np.logspace(-2,4,15)), X, y[dev], cv=4)
    proxy[n]=p
    print(f'  {n:18s} ridge rho {spearmanr(y[dev],p).statistic:.4f}')
ks=list(proxy)
Cm=pd.DataFrame([[spearmanr(proxy[a],proxy[b]).statistic for b in ks] for a in ks],
                index=ks, columns=ks)
print('\npairwise correlation of their predictions:')
print(Cm.round(2).to_string())
# greedy: keep the best, then add whichever is least correlated with everything kept
order=sorted(ks, key=lambda k:-spearmanr(y[dev],proxy[k]).statistic)
SEL=[order[0]]
for _ in range(min(4,len(ks)-1)):
    rest=[k for k in ks if k not in SEL]
    if not rest: break
    SEL.append(min(rest, key=lambda k:max(Cm.loc[k,s] for s in SEL)))
print('\ncarrying forward:', SEL)

## Step 4, TabPFN on the selected encoders

In [ ]:
!pip -q install tabpfn==2.2.1

In [ ]:
from tabpfn import TabPFNRegressor
from sklearn.model_selection import GroupKFold
KW=dict(device=DEVICE, random_state=42, ignore_pretraining_limits=True)
REF={'CYP1A2':0.5881,'CYP2C9':0.7060,'CYP2D6':0.4448,'CYP3A4':0.8115}
RESULTS=[]
for n in SEL:
    a,b=EMB[n]
    X=np.nan_to_num(np.hstack([a,QH,AUX]).astype(np.float32))
    XT=np.nan_to_num(np.hstack([b,QH_TE,AUX_TE]).astype(np.float32))
    print(f'\n{n}  {X.shape[1]} columns', flush=True)
    for iso in ISO:
        yy=meta[f'{iso}_pIC50_direct_inhibition'].to_numpy(float)
        o=np.isfinite(yy); dv=np.flatnonzero(o&~hold); ho=np.flatnonzero(o&hold)
        t0=time.time(); oof=np.full(len(dv),np.nan)
        for tr,va in GroupKFold(5).split(dv,dv,G[dv]):
            oof[va]=TabPFNRegressor(**KW).fit(X[dv[tr]],yy[dv[tr]]).predict(X[dv[va]])
        hp=TabPFNRegressor(**KW).fit(X[dv],yy[dv]).predict(X[ho])
        allr=np.concatenate([dv,ho])
        tp=TabPFNRegressor(**KW).fit(X[allr],yy[allr]).predict(XT)
        np.savez_compressed(OUT/f'fnd_{n}_{iso}.npz', oof=oof, dev=dv, hold=hp, ho=ho, test=tp)
        d=float(spearmanr(yy[dv],oof).statistic); h=float(spearmanr(yy[ho],hp).statistic)
        RESULTS.append(dict(enc=n, iso=iso, dev_spearman=d, holdout_spearman=h, ref=REF[iso]))
        json.dump(RESULTS, open(OUT/'results_nb12.json','w'), indent=2)
        print(f'  {iso}: dev {d:.4f} (our best arm {REF[iso]:.4f}) | holdout {h:.4f} '
              f'({time.time()-t0:.0f}s)', flush=True)
r=pd.DataFrame(RESULTS)
t=r.pivot(index='enc',columns='iso',values='dev_spearman'); t['MACRO']=t.mean(axis=1)
print('\n'+t.round(4).to_string())
print('our TabPFN arm macro 0.6376. These do not need to beat it, only to differ.')
import shutil
from google.colab import files
shutil.make_archive('/content/nb12_results','zip',OUT)
files.download('/content/nb12_results.zip')